# Session 4 — Bandits: an RL-based agent

**DAT 5668 · Module 2, Session 2 · Practice (about 80 minutes)**

**Family this session: RL-based agent.** Repeated numeric choices and a reward. Session 3's LLM-based agent does not fit this job, and this bandit does not fit Session 3's documents.

![A bandit picks one offer](../../assets/bandit.png)

## Run of show
- 10 min — Why this is not Session 3's document problem.
- 55 min — Implement ε-greedy and read the regret.
- 15 min — When a bandit is the wrong tool (and an LLM agent is the wrong tool).

## Business setting
A retailer can show one of three shipping offers each visit. Only one offer is displayed, so you never see the reward of the offers you skipped. That is a multi-armed bandit.

Instructional rewards, not live traffic. True accept rates: Offer A 0.20, Offer B 0.55, Offer C 0.35. The agent does not know these numbers.


In [ ]:
import numpy as np

OFFERS = ["A", "B", "C"]
TRUE_RATE = {"A": 0.20, "B": 0.55, "C": 0.35}
RNG = np.random.default_rng(7)


def customer_accepts(offer):
    """One visit. Reward is 1 if the customer accepts the offer, else 0."""
    return int(RNG.random() < TRUE_RATE[offer])


def best_possible_rate():
    return max(TRUE_RATE.values())


print("Offers", OFFERS, "best rate", best_possible_rate())


## Your work

1. Complete `epsilon_greedy`. With probability `epsilon`, pick a random offer index. Otherwise pick an index of the largest value in `values` (break ties toward the smallest index).
2. Complete the loop body where marked so each visit updates `counts` and `values` with the sample mean.
3. Leave `epsilon` at 0.10, then rerun the last cell with 0.0 and with 0.40 and write one sentence on what changed.


In [ ]:
def epsilon_greedy(values, epsilon, rng):
    """TODO: return an action index (0, 1, or 2)."""
    # TODO
    return 0


def run_bandit(epsilon, n_visits=800):
    values = np.zeros(len(OFFERS))
    counts = np.zeros(len(OFFERS))
    rewards = []
    for _ in range(n_visits):
        index = epsilon_greedy(values, epsilon, RNG)
        offer = OFFERS[index]
        reward = customer_accepts(offer)
        # TODO: update counts[index] and values[index] to the new sample mean.
        rewards.append(reward)
    return values, counts, np.cumsum(rewards)


values, counts, cumulative = run_bandit(0.10)
print("values", np.round(values, 3))
print("counts", counts)
print("accepts", int(cumulative[-1]) if len(cumulative) else 0)


In [ ]:
def regret_so_far(cumulative):
    """Extra accepts the best offer would have collected, in expectation."""
    visits = np.arange(1, len(cumulative) + 1)
    best = best_possible_rate() * visits
    return best - cumulative


def report(epsilon):
    values, counts, cumulative = run_bandit(epsilon)
    if counts.sum() == 0:
        print(f"epsilon={epsilon}: NOT YET — the loop never updated the counts")
        return
    gap = regret_so_far(cumulative)[-1]
    learned = OFFERS[int(np.argmax(values))]
    print(
        f"epsilon={epsilon}: learned best={learned}  counts={counts.astype(int).tolist()}  "
        f"expected regret about {gap:.1f}"
    )
    print("  A finished run learns offer B and still spends some visits on A and C when epsilon is 0.10.")


for epsilon in (0.0, 0.10, 0.40):
    report(epsilon)


## Debrief

1. What goes wrong at epsilon 0 if the first unlucky visits make offer B look bad?
2. Why is this a poor job for the Week 1 refund agent?
3. Name a business test you would **not** run as a bandit, because the cost of one bad pull is too high.


## Fill this with me

After `epsilon_greedy` works, pick an epsilon as a class and rerun. 0 never explores. 1 never settles.


In [ ]:
# FILL WITH THE CLASS. A number from 0 to 1.
class_epsilon = None
if class_epsilon is None:
    print("Agree on an epsilon, type it, and run again.")
else:
    report(class_epsilon)
    print("Say whether this was too shy or too restless.")
